## New Step

In [1]:
import json, requests
from datetime import datetime
import pandas as pd
import numpy as np
from datetime import timedelta
import requests
import time
from langchain_ollama import ChatOllama
import json, time
import json
import requests
import time
import os

In [8]:
# ===================== CELL 1: build HOST-focused scene + store what will be shown to LLM =====================
HOST_CSV = "host_stats_onos.csv"          # <-- your host stats CSV
HOST_META = "host_metadata.json"    # <-- optional (mac->host mapping)
WINDOW_MINUTES = 5
TREND_POINTS = 10
SCENE_LOG = "llm_scene_log.jsonl"
DECISION_LOG = "llm_decision_log.jsonl"

def now_iso(): 
    return datetime.now().isoformat()

def build_host_stats(df, trend_points=10):
    """
    Input: df already filtered to the desired time window
    Output: list of host objects (one per MAC)
    """
    hosts = []
    for mac, g in df.groupby("host_mac"):
        g = g.sort_values("timestamp").tail(trend_points)

        tx_pps = g["tx_pps"].astype(float).values
        rx_pps = g["rx_pps"].astype(float).values

        tx_kbps = g["tx_mbps"].astype(float).values * 1000.0
        rx_kbps = g["rx_mbps"].astype(float).values * 1000.0

        if len(tx_pps) < 2:
            continue

        host_obj = {
            "mac": str(mac),

            "tx_pps_trend": [round(x, 2) for x in tx_pps.tolist()],
            "rx_pps_trend": [round(x, 2) for x in rx_pps.tolist()],

            "tx_kbps_trend": [round(x, 2) for x in tx_kbps.tolist()],
            "rx_kbps_trend": [round(x, 2) for x in rx_kbps.tolist()],

            "tx_pps_mean": round(float(np.mean(tx_pps)), 2),
            "rx_pps_mean": round(float(np.mean(rx_pps)), 2),

            "tx_pps_std": round(float(np.std(tx_pps)), 3),
            "rx_pps_std": round(float(np.std(rx_pps)), 3),

            "tx_pps_max": round(float(np.max(tx_pps)), 2),
            "rx_pps_max": round(float(np.max(rx_pps)), 2),

            "tx_kbps_mean": round(float(np.mean(tx_kbps)), 2),
            "rx_kbps_mean": round(float(np.mean(rx_kbps)), 2),

            "tx_kbps_std": round(float(np.std(tx_kbps)), 3),
            "rx_kbps_std": round(float(np.std(rx_kbps)), 3),

            "tx_kbps_max": round(float(np.max(tx_kbps)), 2),
            "rx_kbps_max": round(float(np.max(rx_kbps)), 2),

            "tx_pps_delta": round(float(tx_pps[-1] - tx_pps[-2]), 2),
            "rx_pps_delta": round(float(rx_pps[-1] - rx_pps[-2]), 2),

            "tx_kbps_delta": round(float(tx_kbps[-1] - tx_kbps[-2]), 2),
            "rx_kbps_delta": round(float(rx_kbps[-1] - rx_kbps[-2]), 2),
        }

        hosts.append(host_obj)

    return hosts

def build_host_scene_and_prompt(model_name="gpt-oss:20B", last_k_decisions=10):
    # ---- read host stats ----
    df = pd.read_csv(HOST_CSV)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    if df.empty:
        return None, None

    # ---- filter last WINDOW_MINUTES ----
    t_end = df["timestamp"].max()
    t_min = t_end - pd.Timedelta(minutes=WINDOW_MINUTES)
    dfw = df[df["timestamp"] >= t_min].copy()
    if dfw.empty:
        return None, None
    
    # ---- metadata ----
    try:
        with open(HOST_META, "r", encoding="utf-8") as f:
            host_meta_obj = json.load(f)
    except FileNotFoundError:
        host_meta_obj = {"note": f"{HOST_META} not found"}

    # ---- build host stats via separate function ----
    hosts = build_host_stats(dfw, trend_points=TREND_POINTS)

    #///// filter //////
    hosts = [
        {
            "mac": h["mac"],
            "tx_pps_trend": h["tx_pps_trend"],
            "rx_pps_trend": h["rx_pps_trend"],
            "tx_kbps_trend": h["tx_kbps_trend"],
            "rx_kbps_trend": h["rx_kbps_trend"]
        }
        for h in hosts
    ] 

    # ---- last K decisions ----
    last_decisions = []
    try:
        with open(DECISION_LOG, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    last_decisions.append(json.loads(line))
        last_decisions = last_decisions[-last_k_decisions:]
    except FileNotFoundError:
        last_decisions = []

    # ---- HOST-level scene (what we show to LLM) ----
    scene = {
        "ts": now_iso(),
        "window_minutes": WINDOW_MINUTES,
        "trend_points": TREND_POINTS,
        "window_end_time": str(t_end),
        "counts": {
            "unique_host_macs": int(len(hosts)),
            "raw_rows_in_window": int(len(df)),
        },
        "host_metadata": host_meta_obj,
        "host_stats": hosts,
        "last_decisions": last_decisions
    }
    prompt = f"""
You are an SDN network anomaly detection function. You need to decide on the below task.

Task:
Select host MAC addresses that show DDoS-like behavior.

DDoS-like indicators:
- very high tx_pps_trend or rx_pps_trend
- sudden spike in tx_pps_trend or rx_pps_trend
- sustained high packet rate across multiple trend points
- high PPS with relatively low or moderate kbps
- clear deviation from most other hosts in the same window

Do NOT require both tx and rx to be abnormal.
A host may be suspicious from either tx-side or rx-side behavior alone.

If no host shows DDoS-like behavior, choose do_nothing.

Return ONLY valid JSON.

Output Schema:
{{
 "decision": "ip_shuffle | do_nothing",
 "macs_to_shuffle": ["MAC_ADDRESS"],
 "confidence": 0.0,
 "observation": [
   {{"mac":"MAC_ADDRESS","reason":"short_reason"}}
 ]
}}

Rules:
- Select at least 2 MACs for ip_shuffle.
- Do NOT explain anything.
- Do NOT repeat the input.
- Only return JSON.
- JSON must start with '{{' and end with '}}'.

You have to observe the host stats and output the finding. You must follow the Rules and must respond in json file.
host_stats:
{json.dumps(hosts)}

Return JSON.
""".strip()

    # we don;t need decision now
    # # ---- log exactly what LLM saw ----
    # with open(SCENE_LOG, "a", encoding="utf-8") as f:
    #     f.write(json.dumps({
    #         "ts": now_iso(),
    #         "model": model_name,
    #         "scene": scene,
    #         "prompt": prompt
    #     }) + "\n")

    print(prompt)
    return scene, prompt


import json, time
from datetime import datetime
import numpy as np
import pandas as pd

LINK_CSV   = "link_stats_onos.csv"
WINDOW_MINUTES = 5
TREND_POINTS   = 10

SCENE_LOG    = "llm_scene_log_link.jsonl"
DECISION_LOG = "llm_decision_log_link.jsonl"

def now_iso():
    return datetime.now().isoformat()

def build_link_stats(df, trend_points=10):
    """
    Input: df already filtered to the desired time window
    Output: list of link objects (one per link_id)
    """
    links = []

    for link_id, g in df.groupby("link_id"):
        g = g.sort_values("timestamp").tail(trend_points)

        rx_pps = g["rx_pps"].astype(float).values
        tx_pps = g["tx_pps"].astype(float).values

        rx_kbps = g["rx_mbps"].astype(float).values * 1000.0
        tx_kbps = g["tx_mbps"].astype(float).values * 1000.0

        if len(rx_pps) < 2:
            continue

        link_obj = {
            "link_id": str(link_id),

            "rx_pps_trend": [round(x, 2) for x in rx_pps.tolist()],
            "tx_pps_trend": [round(x, 2) for x in tx_pps.tolist()],

            "rx_kbps_trend": [round(x, 2) for x in rx_kbps.tolist()],
            "tx_kbps_trend": [round(x, 2) for x in tx_kbps.tolist()],

            "rx_pps_mean": round(float(np.mean(rx_pps)), 2),
            "tx_pps_mean": round(float(np.mean(tx_pps)), 2),

            "rx_pps_std": round(float(np.std(rx_pps)), 3),
            "tx_pps_std": round(float(np.std(tx_pps)), 3),

            "rx_pps_max": round(float(np.max(rx_pps)), 2),
            "tx_pps_max": round(float(np.max(tx_pps)), 2),

            "rx_kbps_mean": round(float(np.mean(rx_kbps)), 2),
            "tx_kbps_mean": round(float(np.mean(tx_kbps)), 2),

            "rx_kbps_std": round(float(np.std(rx_kbps)), 3),
            "tx_kbps_std": round(float(np.std(tx_kbps)), 3),

            "rx_kbps_max": round(float(np.max(rx_kbps)), 2),
            "tx_kbps_max": round(float(np.max(tx_kbps)), 2),

            "rx_pps_delta": round(float(rx_pps[-1] - rx_pps[-2]), 2),
            "tx_pps_delta": round(float(tx_pps[-1] - tx_pps[-2]), 2),

            "rx_kbps_delta": round(float(rx_kbps[-1] - rx_kbps[-2]), 2),
            "tx_kbps_delta": round(float(tx_kbps[-1] - tx_kbps[-2]), 2),
        }

        links.append(link_obj)

    return links

def build_link_scene_and_prompt(model_name="llama3", last_k_decisions=10):
    df = pd.read_csv(LINK_CSV)
    df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["timestamp"])

    if df.empty:
        return None, None

    # --- filter last WINDOW_MINUTES ---
    t_end = df["timestamp"].max()
    t_min = t_end - pd.Timedelta(minutes=WINDOW_MINUTES)
    dfw = df[df["timestamp"] >= t_min].copy()

    # 👉 ADDED THIS
    global last_ts_scene 
    last_ts_scene = t_end

    if dfw.empty:
        return None, None

    # --- build link stats ---
    links = build_link_stats(dfw, trend_points=TREND_POINTS)

    # --- filter only link_id + trends (like hosts) ---
    links = [
        {
            "link_id": l["link_id"],
            "rx_pps_trend": l["rx_pps_trend"],
            "tx_pps_trend": l["tx_pps_trend"],
            "rx_kbps_trend": l["rx_kbps_trend"],
            "tx_kbps_trend": l["tx_kbps_trend"]
        }
        for l in links
    ]


    # --- last K decisions ---
    last_decisions = []
    try:
        with open(DECISION_LOG, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    last_decisions.append(json.loads(line))
        last_decisions = last_decisions[-last_k_decisions:]
    except FileNotFoundError:
        last_decisions = []

    scene = {
        "ts": now_iso(),
        "window_minutes": WINDOW_MINUTES,
        "trend_points": TREND_POINTS,
        "window_end_time": str(t_end),
        "counts": {
            "unique_links": int(len(links)),
            "raw_rows_in_window": int(len(dfw)),
        },
        "link_stats": links,
        "last_decisions": last_decisions
    }

    prompt = f"""
You are an SDN network anomaly detection function for LINKS.

Task:
Select link_id values that show abnormal congestion (rising load / spikes).

Indicators of abnormal behavior:
- rx_pps_trend rising / spiking
- tx_pps_trend rising / spiking
- rx_kbps_trend rising / spiking
- tx_kbps_trend rising / spiking

Interpretation rules:
- A spike means a sharp increase between consecutive trend points.
- Rising means a clear upward pattern across multiple trend points.
- Prefer sustained increases over isolated one-point spikes.
- High kbps is stronger evidence of bandwidth stress than PPS alone.
- If both PPS and kbps rise together, that is a stronger congestion signal.
- Deviation means the link is clearly higher than most other links in the same window.

If no link meets these conditions, choose do_nothing.

Return ONLY valid JSON.

Output Schema:
{{
 "decision": "reroute | do_nothing",
 "links_to_avoid": ["LINK_ID"],
 "confidence": 0.0,
 "observation": [
   {{"link_id":"LINK_ID","reason":"short_reason"}}
 ]
}}

Rules:
- Do NOT explain anything.
- Do NOT repeat the input.
- Only return JSON.
- JSON must start with '{{' and end with '}}'.

link_stats:
{json.dumps(links)}

Return JSON.
""".strip()

    # we do not need link scene captured now
    # with open(SCENE_LOG, "a", encoding="utf-8") as f:
    #     f.write(json.dumps({
    #         "ts": now_iso(),
    #         "model": model_name,
    #         "scene": scene,
    #         "prompt": prompt
    #     }) + "\n")

    print(prompt)
    return scene, prompt

In [1]:
import requests, time, json

CLOUD_URL = "https://ollama.com/api/chat"
API_KEY = "23fbf0f676584a7983158ded37540f2c.9C4Aw8pI8jXQlC1vDCGIF7nb"
MODEL_NAME = "gpt-oss:20b-cloud"

def call_cloud_llm(prompt):

    payload = {
        "model": MODEL_NAME,
        "messages": [
            {
                "role": "system",
                "content": "Return ONLY valid JSON. No text."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False,
        "options": {
            "temperature": 0,
            "top_p": 0.9
        }
    }

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    start = time.time()

    r = requests.post(CLOUD_URL, headers=headers, json=payload, timeout=300)
    r.raise_for_status()

    latency = time.time() - start

    out = r.json().get("message", {}).get("content", "").strip()

    return out, latency

def call_cloud_llm_judge(prompt,model_name_1):

    payload = {
        "model": model_name_1,
        "messages": [
            {
                "role": "system",
                "content": "Return ONLY valid JSON. No text."
            },
            {
                "role": "user",
                "content": prompt
            }
        ],
        "stream": False,
        "options": {
            "temperature": 0,
            "top_p": 0.9
        }
    }

    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }

    start = time.time()

    r = requests.post(CLOUD_URL, headers=headers, json=payload, timeout=300)
    r.raise_for_status()

    latency = time.time() - start

    out = r.json().get("message", {}).get("content", "").strip()

    return out, latency

In [11]:
# ////////hosts

scene, prompt = build_host_scene_and_prompt()

out, latency = call_cloud_llm(prompt)

print(out)
print("Latency for Host:", latency)


# ////////links


scene2, prompt2 = build_link_scene_and_prompt()

out2, latency2 = call_cloud_llm(prompt2)

print(out2)
print("Latency for Links:", latency2)

You are an SDN network anomaly detection function. You need to decide on the below task.

Task:
Select host MAC addresses that show DDoS-like behavior.

DDoS-like indicators:
- very high tx_pps_trend or rx_pps_trend
- sudden spike in tx_pps_trend or rx_pps_trend
- sustained high packet rate across multiple trend points
- high PPS with relatively low or moderate kbps
- clear deviation from most other hosts in the same window

Do NOT require both tx and rx to be abnormal.
A host may be suspicious from either tx-side or rx-side behavior alone.

If no host shows DDoS-like behavior, choose do_nothing.

Return ONLY valid JSON.

Output Schema:
{
 "decision": "ip_shuffle | do_nothing",
 "macs_to_shuffle": ["MAC_ADDRESS"],
 "confidence": 0.0,
 "observation": [
   {"mac":"MAC_ADDRESS","reason":"short_reason"}
 ]
}

Rules:
- Select at least 2 MACs for ip_shuffle.
- Do NOT explain anything.
- Do NOT repeat the input.
- Only return JSON.
- JSON must start with '{' and end with '}'.

You have to obs

In [49]:
# # ===================== COMPACT FINAL FUSION CELL =====================

# import json, re, time
# from datetime import datetime
# import pandas as pd

# def get_flagged_entities_with_trends(out1, out2):
#     """
#     Reads host/link LLM outputs, extracts flagged MACs and link_ids,
#     then fetches last WINDOW_MINUTES of trend data only for those entities.

#     Returns:
#         host_report
#         link_report
#         flagged_macs
#         flagged_links
#         flagged_host_stats
#         flagged_link_stats
#     """

#     # ---------- parse LLM outputs ----------
#     host_report = json.loads(out1) if isinstance(out1, str) else out1
#     link_report = json.loads(out2) if isinstance(out2, str) else out2

#     flagged_macs = host_report.get("macs_to_shuffle", []) if host_report.get("decision") == "ip_shuffle" else []
#     flagged_links = link_report.get("links_to_avoid", []) if link_report.get("decision") == "reroute" else []

#     flagged_macs_set = set(flagged_macs)
#     flagged_links_set = set(flagged_links)

#     # ---------- host side ----------
#     flagged_host_stats = []
#     hdf = pd.read_csv(HOST_CSV)
#     hdf["timestamp"] = pd.to_datetime(hdf["timestamp"], errors="coerce")
#     hdf = hdf.dropna(subset=["timestamp"])

#     if not hdf.empty and flagged_macs_set:
#         h_end = hdf["timestamp"].max()
#         h_min = h_end - pd.Timedelta(minutes=WINDOW_MINUTES)
#         hdfw = hdf[hdf["timestamp"] >= h_min].copy()

#         if not hdfw.empty:
#             all_hosts = build_host_stats(hdfw, trend_points=TREND_POINTS)
#             flagged_host_stats = [
#                 {
#                     "mac": h["mac"],
#                     "tx_pps_trend": h["tx_pps_trend"],
#                     "rx_pps_trend": h["rx_pps_trend"],
#                     "tx_kbps_trend": h["tx_kbps_trend"],
#                     "rx_kbps_trend": h["rx_kbps_trend"]
#                 }
#                 for h in all_hosts if h["mac"] in flagged_macs_set
#             ]

#     # ---------- link side ----------
#     flagged_link_stats = []
#     ldf = pd.read_csv(LINK_CSV)
#     ldf["timestamp"] = pd.to_datetime(ldf["timestamp"], errors="coerce")
#     ldf = ldf.dropna(subset=["timestamp"])

#     if not ldf.empty and flagged_links_set:
#         l_end = ldf["timestamp"].max()
#         l_min = l_end - pd.Timedelta(minutes=WINDOW_MINUTES)
#         ldfw = ldf[ldf["timestamp"] >= l_min].copy()

#         if not ldfw.empty:
#             all_links = build_link_stats(ldfw, trend_points=TREND_POINTS)
#             flagged_link_stats = [
#                 {
#                     "link_id": l["link_id"],
#                     "rx_pps_trend": l["rx_pps_trend"],
#                     "tx_pps_trend": l["tx_pps_trend"],
#                     "rx_kbps_trend": l["rx_kbps_trend"],
#                     "tx_kbps_trend": l["tx_kbps_trend"]
#                 }
#                 for l in all_links if l["link_id"] in flagged_links_set
#             ]

#     return (
#         host_report,
#         link_report,
#         flagged_macs,
#         flagged_links,
#         flagged_host_stats,
#         flagged_link_stats
#     )

# host_report, link_report, flagged_macs, flagged_links, flagged_host_stats, flagged_link_stats = \
#     get_flagged_entities_with_trends(out, out2)

# scene = {
#     "host_report": host_report,
#     "link_report": link_report,
#     "flagged_host_stats": flagged_host_stats,
#     "flagged_link_stats": flagged_link_stats,
#     "policy_order": ["do_nothing", "rrm", "ip", "both"]
# }

# prompt = f"""
# You are the final SDN mitigation decision function.

# You are given:
# 1. a host anomaly report,
# 2. a link anomaly report,
# 3. the recent 5-minute trend data only for the hosts and links that were flagged.
# 4. Select at least two hosts only when routing-related action is chosen.

# Your job:
# - Recheck the flagged host and link trends.
# - Decide which flagged hosts and links remain true final candidates.
# - Decide the final mitigation action.

# Defense order:
# do_nothing < rrm < ip < both

# Meaning:
# - do_nothing = lowest cost, weakest response
# - rrm = stronger than do_nothing, lower cost than ip
# - ip = stronger than rrm, higher cost than rrm
# - both = strongest response, highest cost

# Policy:
# - Later actions are stronger defenses.
# - Later actions also cost more.
# - Choose the lowest-cost action that is still sufficient for the verified severity.
# - Do not escalate to a stronger action unless the trends clearly justify it.
# - Use both only when host-side abnormality and link-side abnormality are both clearly strong after recheck.
# - If the reported abnormality is weak or unsupported after recheck, choose do_nothing.

# Guidance:
# - Prefer rrm when the main problem is link congestion or suspicious link rise.
# - Prefer ip when the main problem is host-side suspicious traffic growth and host evidence is stronger than link-only evidence.
# - Prefer both only when flagged hosts and flagged links both remain strongly abnormal and combined defense is justified.
# - Return only final candidates that are still supported after recheck.
# - If no candidate remains supported, return empty target lists and choose do_nothing.

# Return ONLY valid JSON.

# Output schema:
# {{
#   "final_decision": "do_nothing | rrm | ip | both",
#   "final_macs": ["MAC_ADDRESS"],
#   "final_links": ["LINK_ID"],
#   "confidence": 0.0,
#   "severity": "low | medium | high | critical",
#   "observation": [
#     {{"type":"host","id":"MAC_ADDRESS","reason":"short_reason"}},
#     {{"type":"link","id":"LINK_ID","reason":"short_reason"}}
#   ]
# }}

# Rules:
# - Output JSON only.
# - No explanation outside JSON.
# - JSON must start with '{{' and end with '}}'.
# - Keep only rechecked and supported final candidates.
# - If action is do_nothing, return empty lists for final_macs and final_links.

# Input:
# {json.dumps(scene, ensure_ascii=False)}

# Return JSON only.
# """.strip()

# final_out, final_latency = call_cloud_llm(prompt)
# final_obj = json.loads(final_out)

# print("FINAL RAW:", final_out)
# print("FINAL JSON:", json.dumps(final_obj, indent=2))
# print("FINAL LATENCY:", final_latency)


In [28]:
# ===================== COMPACT FINAL FUSION CELL =====================

import json, re, time
from datetime import datetime
import pandas as pd

def get_flagged_entities_with_trends(out1, out2):
    """
    Reads host/link LLM outputs, extracts flagged MACs and link_ids,
    then fetches last WINDOW_MINUTES of trend data only for those entities.

    Returns:
        host_report
        link_report
        flagged_macs
        flagged_links
        flagged_host_stats
        flagged_link_stats
    """

    # ---------- parse LLM outputs ----------
    host_report = json.loads(out1) if isinstance(out1, str) else out1
    link_report = json.loads(out2) if isinstance(out2, str) else out2

    flagged_macs = host_report.get("macs_to_shuffle", []) if host_report.get("decision") == "ip_shuffle" else []
    flagged_links = link_report.get("links_to_avoid", []) if link_report.get("decision") == "reroute" else []

    flagged_macs_set = set(flagged_macs)
    flagged_links_set = set(flagged_links)

    # ---------- host side ----------
    flagged_host_stats = []
    hdf = pd.read_csv(HOST_CSV)
    hdf["timestamp"] = pd.to_datetime(hdf["timestamp"], errors="coerce")
    hdf = hdf.dropna(subset=["timestamp"])

    if not hdf.empty and flagged_macs_set:
        h_end = hdf["timestamp"].max()
        h_min = h_end - pd.Timedelta(minutes=WINDOW_MINUTES)
        hdfw = hdf[hdf["timestamp"] >= h_min].copy()

        if not hdfw.empty:
            all_hosts = build_host_stats(hdfw, trend_points=TREND_POINTS)
            flagged_host_stats = [
                {
                    "mac": h["mac"],
                    "tx_pps_trend": h["tx_pps_trend"],
                    "rx_pps_trend": h["rx_pps_trend"],
                    "tx_kbps_trend": h["tx_kbps_trend"],
                    "rx_kbps_trend": h["rx_kbps_trend"]
                }
                for h in all_hosts if h["mac"] in flagged_macs_set
            ]

    # ---------- link side ----------
    flagged_link_stats = []
    ldf = pd.read_csv(LINK_CSV)
    ldf["timestamp"] = pd.to_datetime(ldf["timestamp"], errors="coerce")
    ldf = ldf.dropna(subset=["timestamp"])

    if not ldf.empty and flagged_links_set:
        l_end = ldf["timestamp"].max()
        l_min = l_end - pd.Timedelta(minutes=WINDOW_MINUTES)
        ldfw = ldf[ldf["timestamp"] >= l_min].copy()

        if not ldfw.empty:
            all_links = build_link_stats(ldfw, trend_points=TREND_POINTS)
            flagged_link_stats = [
                {
                    "link_id": l["link_id"],
                    "rx_pps_trend": l["rx_pps_trend"],
                    "tx_pps_trend": l["tx_pps_trend"],
                    "rx_kbps_trend": l["rx_kbps_trend"],
                    "tx_kbps_trend": l["tx_kbps_trend"]
                }
                for l in all_links if l["link_id"] in flagged_links_set
            ]

    return (
        host_report,
        link_report,
        flagged_macs,
        flagged_links,
        flagged_host_stats,
        flagged_link_stats
    )

host_report, link_report, flagged_macs, flagged_links, flagged_host_stats, flagged_link_stats = \
    get_flagged_entities_with_trends(out, out2)

scene = {
    "host_report": host_report,
    "link_report": link_report,
    "flagged_host_stats": flagged_host_stats,
    "flagged_link_stats": flagged_link_stats,
    "policy_order": ["do_nothing", "rrm", "ip", "both"]
}

prompt = f"""
You are the final SDN mitigation decision function.

You are given:
1. a host anomaly report,
2. a link anomaly report,
3. the recent 5-minute trend data only for the hosts and links that were flagged.
4. select atleast two host and one link

Your job:
- Recheck the flagged host and link trends.
- Decide which flagged hosts and links remain true final candidates.
- Decide the final mitigation action.

Defense order:
do_nothing < rrm < ip < both

Meaning:
- do_nothing = lowest cost, weakest response
- rrm = stronger than do_nothing, lower cost than ip
- ip = stronger than rrm, higher cost than rrm
- both = strongest response, highest cost

Mitigation selection rule:
- First verify whether any real attack-like abnormality remains after recheck.
- Then choose the cheapest sufficient response.
- Do not use ip as a default response; reserve it for clearly supported host-side threats.

Policy:
- Later actions are stronger defenses.
- Later actions also cost more.
- Choose the lowest-cost action that is still sufficient for the verified severity.
- Do not escalate to a stronger action unless the trends clearly justify it.
- If the rechecked host/link trends stay flat or near baseline (roughly 1.0-1.3) and do not show a clear recent spike or sustained rise, treat them as normal and choose do_nothing.
- Use both only when host-side abnormality and link-side abnormality are both clearly strong after recheck.
- If the reported abnormality is weak or unsupported after recheck, choose do_nothing.

Guidance:
- Prefer rrm when the main problem is link congestion or suspicious link rise.
- Prefer ip when the main problem is host-side suspicious traffic growth and host evidence is stronger than link-only evidence.
- Prefer both only when flagged hosts and flagged links both remain strongly abnormal and combined defense is justified.
- Return only final candidates that are still supported after recheck.
- If no candidate remains supported, return empty target lists and choose do_nothing.

Return ONLY valid JSON.


Output schema:
{{
  "final_decision": "do_nothing | rrm | ip | both",
  "final_macs": ["MAC_ADDRESS"],
  "final_links": ["LINK_ID"],
  "confidence": 0.0,
  "severity": "low | medium | high | critical",
  "observation": [
    {{"type":"host","id":"MAC_ADDRESS","reason":"short_reason"}},
    {{"type":"link","id":"LINK_ID","reason":"short_reason"}}
  ]
}}

Rules:
- Output JSON only.
- No explanation outside JSON.
- JSON must start with '{{' and end with '}}'.
- Keep only rechecked and supported final candidates.
- If action is do_nothing, return empty lists for final_macs and final_links.

Input:
{json.dumps(scene, ensure_ascii=False)}

Return JSON only.
""".strip()

final_out, final_latency = call_cloud_llm(prompt)
final_obj = json.loads(final_out)

print("FINAL RAW:", final_out)
print("FINAL JSON:", json.dumps(final_obj, indent=2))
print("FINAL LATENCY:", final_latency)


FINAL RAW: {"final_decision":"do_nothing","final_macs":[],"final_links":[],"confidence":0.0,"severity":"low","observation":[]}
FINAL JSON: {
  "final_decision": "do_nothing",
  "final_macs": [],
  "final_links": [],
  "confidence": 0.0,
  "severity": "low",
  "observation": []
}
FINAL LATENCY: 3.0179624557495117


In [13]:
scenex=scene

In [52]:
# import pandas as pd

# def path_selector(final_obj, hoplist_csv):
#     """
#     Return safe candidate paths only between the selected hosts in final_obj,
#     while avoiding blocked links in both directions, and keeping only one
#     direction per host pair (e.g., keep h1->h2, drop h2->h1).

#     hoplist.csv columns:
#         host1, host2, option_number, hop_count, src_mac, dst_mac, path
#     """

#     def normalize_link(link_str):
#         left, right = [x.strip() for x in str(link_str).split("->")]
#         return tuple(sorted([left, right]))

#     selected_macs = set(final_obj.get("final_macs", []))
#     blocked_links = set(
#         normalize_link(x.strip()) for x in final_obj.get("final_links", [])
#     )

#     df = pd.read_csv(
#         hoplist_csv,
#         header=None,
#         names=["host1", "host2", "option_number", "hop_count", "src_mac", "dst_mac", "path"]
#     )

#     # keep only rows where BOTH endpoints are among the selected hosts
#     pair_df = df[
#         (df["src_mac"].isin(selected_macs)) & (df["dst_mac"].isin(selected_macs))
#     ].copy()

#     if pair_df.empty:
#         return []

#     def path_is_safe(path_str):
#         links = [x.strip() for x in str(path_str).split(",")]
#         norm_links = [normalize_link(link) for link in links]
#         return all(link not in blocked_links for link in norm_links)

#     pair_df["is_safe"] = pair_df["path"].apply(path_is_safe)
#     safe_df = pair_df[pair_df["is_safe"]].copy()

#     if safe_df.empty:
#         return []

#     # treat h1->h2 and h2->h1 as the same host pair
#     safe_df["pair_key"] = safe_df.apply(
#         lambda r: tuple(sorted([r["host1"], r["host2"]])),
#         axis=1
#     )

#     # keep only one direction per pair; alphabetical order keeps h1->h2 over h2->h1
#     safe_df = safe_df.sort_values(
#         ["pair_key", "host1", "host2", "hop_count", "option_number"]
#     )
#     safe_df = safe_df.drop_duplicates(subset=["pair_key"], keep="first")

#     candidates = []
#     for _, row in safe_df.iterrows():
#         candidates.append({
#             "host1": row["host1"],
#             "host2": row["host2"],
#             "option_number": int(row["option_number"]),
#             "hop_count": int(row["hop_count"]),
#             "src_mac": row["src_mac"],
#             "dst_mac": row["dst_mac"],
#             "path": row["path"]
#         })

#     return candidates

In [ ]:
# import pandas as pd

# def path_selector(final_obj, hoplist_csv):
#     """
#     Return safe candidate paths only between the selected hosts in final_obj,
#     while avoiding blocked links in both directions, and keeping only one
#     direction per host pair (e.g., keep h1->h2, drop h2->h1).

#     hoplist.csv columns:
#         host1, host2, option_number, hop_count, src_mac, dst_mac, path
#     """

#     def normalize_link(link_str):
#         left, right = [x.strip() for x in str(link_str).split("->")]
#         return tuple(sorted([left, right]))

#     selected_macs = set(final_obj.get("final_macs", []))
#     blocked_links = set(
#         normalize_link(x.strip()) for x in final_obj.get("final_links", [])
#     )

#     df = pd.read_csv(
#         hoplist_csv,
#         header=None,
#         names=["host1", "host2", "option_number", "hop_count", "src_mac", "dst_mac", "path"]
#     )

#     # keep only rows where BOTH endpoints are among the selected hosts
#     pair_df = df[
#         (df["src_mac"].isin(selected_macs)) & (df["dst_mac"].isin(selected_macs))
#     ].copy()

#     if pair_df.empty:
#         return []

#     def blocked_links_used(path_str):
#         links = [x.strip() for x in str(path_str).split(",")]
#         norm_links = [normalize_link(link) for link in links]
#         return sum(1 for link in norm_links if link in blocked_links)

#     pair_df["blocked_used"] = pair_df["path"].apply(blocked_links_used)

#     # treat h1->h2 and h2->h1 as the same host pair
#     pair_df["pair_key"] = pair_df.apply(
#         lambda r: tuple(sorted([r["host1"], r["host2"]])),
#         axis=1
#     )

#     pair_df = pair_df.sort_values(
#         ["pair_key", "blocked_used", "hop_count", "option_number", "host1", "host2"]
#     )

#     best_df = pair_df.drop_duplicates(subset=["pair_key"], keep="first")

#     candidates = []
#     for _, row in best_df.iterrows():
#         candidates.append({
#             "host1": row["host1"],
#             "host2": row["host2"],
#             "option_number": int(row["option_number"]),
#             "hop_count": int(row["hop_count"]),
#             "src_mac": row["src_mac"],
#             "dst_mac": row["dst_mac"],
#             "path": row["path"]
#         })

#     return candidates

In [14]:

#new ones handle link rerouting even if one or no hosts provided

def path_selector(final_obj, hoplist_csv):
    """
    Returns only what do_rrm() needs:
    - host1
    - host2
    - option_number

    Rules:
    - 2+ hosts: original behavior
    - 1 host: anchor host, one pair per blocked link
    - 0 host: one active pair per blocked link
    """

    print("\n running path selector")

    def normalize_link(link_str):
        left, right = [x.strip() for x in str(link_str).split("->")]
        return tuple(sorted([left, right]))

    def parse_path_links(path_str):
        return [x.strip() for x in str(path_str).split(",") if str(x).strip()]

    selected_macs = [str(x).upper() for x in final_obj.get("final_macs", [])]
    blocked_links = [
        normalize_link(str(x).strip())
        for x in final_obj.get("final_links", [])
        if str(x).strip()
    ]

    if not blocked_links:
        return []

    df = pd.read_csv(
        hoplist_csv,
        header=None,
        names=["host1", "host2", "option_number", "hop_count", "src_mac", "dst_mac", "path"]
    )

    df["src_mac"] = df["src_mac"].astype(str).str.upper()
    df["dst_mac"] = df["dst_mac"].astype(str).str.upper()
    df["option_number"] = df["option_number"].astype(int)

    df["pair_key"] = df.apply(
        lambda r: tuple(sorted([str(r["host1"]), str(r["host2"])])),
        axis=1
    )

    df["norm_path_links"] = df["path"].apply(
        lambda p: [normalize_link(x) for x in parse_path_links(p)]
    )

    # route_history directly loaded here
    activity_map = {}
    try:
        rh = pd.read_csv("route_history.csv")

        rh["pair_key"] = rh.apply(
            lambda r: tuple(sorted([str(r["host_a"]), str(r["host_b"])])),
            axis=1
        )

        def activity_score(hist):
            vals = []
            for x in str(hist).split(","):
                x = x.strip()
                if not x:
                    continue
                try:
                    vals.append(int(x))
                except:
                    vals.append(0)
            return sum(1 for v in vals if v != 0)

        rh["activity_score"] = rh["history"].apply(activity_score)
        activity_map = dict(zip(rh["pair_key"], rh["activity_score"]))

    except Exception as e:
        print("[WARN] could not read route history:", e)

    def blocked_used(norm_links):
        return sum(1 for lk in norm_links if lk in blocked_links)

    def choose_best_option(pair_df):
        pair_df = pair_df.copy()
        pair_df["blocked_used"] = pair_df["norm_path_links"].apply(blocked_used)

        safe_df = pair_df[pair_df["blocked_used"] == 0].copy()
        if not safe_df.empty:
            safe_df = safe_df.sort_values(["hop_count", "option_number"])
            return safe_df.iloc[0]

        pair_df = pair_df.sort_values(["blocked_used", "hop_count", "option_number"])
        return pair_df.iloc[0]

    # --------------------------------------------------
    # MODE A: 2+ hosts
    # --------------------------------------------------
    if len(selected_macs) >= 2:
        selected_macs_set = set(selected_macs)

        pair_df = df[
            (df["src_mac"].isin(selected_macs_set)) &
            (df["dst_mac"].isin(selected_macs_set))
        ].copy()

        if pair_df.empty:
            return []

        pair_df["blocked_used"] = pair_df["norm_path_links"].apply(blocked_used)
        pair_df = pair_df.sort_values(
            ["pair_key", "blocked_used", "hop_count", "option_number"]
        )

        best_df = pair_df.drop_duplicates(subset=["pair_key"], keep="first")

        return [
            {
                "host1": row["host1"],
                "host2": row["host2"],
                "option_number": int(row["option_number"])
            }
            for _, row in best_df.iterrows()
        ]

    # --------------------------------------------------
    # MODE B/C: 1 host anchor OR 0 host
    # one pair per blocked link
    # --------------------------------------------------
    anchor_mac = selected_macs[0] if len(selected_macs) == 1 else None

    chosen_pairs = set()
    candidates = []

    for blk in blocked_links:
        link_df = df[df["norm_path_links"].apply(lambda links: blk in links)].copy()

        if link_df.empty:
            continue

        # prefer anchor host if one exists
        if anchor_mac:
            anchor_df = link_df[
                (link_df["src_mac"] == anchor_mac) | (link_df["dst_mac"] == anchor_mac)
            ].copy()

            if not anchor_df.empty:
                link_df = anchor_df

        pair_rows = []
        for pair_key, g in link_df.groupby("pair_key"):
            act = activity_map.get(pair_key, 0)
            best_row = choose_best_option(g)
            best_blocked = blocked_used(best_row["norm_path_links"])

            pair_rows.append({
                "pair_key": pair_key,
                "activity_score": act,
                "best_blocked": best_blocked,
                "best_hop_count": int(best_row["hop_count"])
            })

        if not pair_rows:
            continue

        rank_df = pd.DataFrame(pair_rows).sort_values(
            ["activity_score", "best_blocked", "best_hop_count"],
            ascending=[False, True, True]
        )

        picked_pair = None
        for _, rr in rank_df.iterrows():
            if rr["pair_key"] not in chosen_pairs:
                picked_pair = rr["pair_key"]
                break

        if picked_pair is None:
            picked_pair = rank_df.iloc[0]["pair_key"]

        chosen_pairs.add(picked_pair)

        pair_df = link_df[link_df["pair_key"] == picked_pair].copy()
        best = choose_best_option(pair_df)

        candidates.append({
            "host1": best["host1"],
            "host2": best["host2"],
            "option_number": int(best["option_number"])
        })

    return candidates
#new ones handle link rerouting even if one or no hosts provided

In [15]:
final_obj

{'final_decision': 'do_nothing',
 'final_macs': [],
 'final_links': [],
 'confidence': 0.9,
 'severity': 'low',
 'observation': [{'type': 'host',
   'id': '00:00:00:00:00:1B',
   'reason': 'transient tx spike'},
  {'type': 'host', 'id': '00:00:00:00:00:25', 'reason': 'transient tx spike'}]}

In [16]:
candidates = path_selector(final_obj, "hop_list.csv")

for c in candidates:
    print(c)


 running path selector


In [56]:
# import subprocess
# import random


# TOPOLOGY_FILE = "topology_s10.txt"
# IP_SHUFFLER_SCRIPT = "./ip_shuffle.py"
# RRM_SCRIPT = "./rrm.py"


# def parse_topology_hosts(topology_file):
#     """
#     Reads host entries like:
#       h1, 10.0.0.1/24, 00:00:00:00:00:01

#     Returns:
#       mac_to_host = {
#           "00:00:00:00:00:01": "h1",
#           ...
#       }
#     """
#     mac_to_host = {}

#     with open(topology_file, "r") as f:
#         for raw in f:
#             line = raw.strip()

#             if not line or line.startswith("#"):
#                 continue

#             parts = [p.strip() for p in line.split(",")]

#             if len(parts) == 3 and parts[0].startswith("h") and "/" in parts[1] and ":" in parts[2]:
#                 host = parts[0]
#                 mac = parts[2].upper()
#                 mac_to_host[mac] = host

#     return mac_to_host


# MAC_TO_HOST = parse_topology_hosts(TOPOLOGY_FILE)


# def resolve_hosts_from_macs(macs):
#     hosts = []

#     for mac in macs:
#         host = MAC_TO_HOST.get(mac.upper())
#         if host:
#             hosts.append(host)
#         else:
#             print(f"[WARN] MAC not found in topology: {mac}")

#     return hosts


# def build_ip_octets_for_hosts(hosts):
#     """
#     Example mapping:
#       h1 -> 10
#       h2 -> 20
#       h3 -> 30
#     Change this logic if you want another rule.
#     """
#     octets = []

#     for h in hosts:
#         try:
#             num = int(h[1:])
#             octet = str(num * 10)
#             octets.append(octet)
#         except Exception:
#             print(f"[WARN] Could not derive octet for host {h}")
#             octets.append(str(random.randint(10, 200)))

#     return octets


# def do_ip_shuffle(macs):
#     """
#     Runs one command like:
#       sudo python3 ip_hopper.py --host "h1,h2" --ips "10;20"
#     """
#     print("[IP SHUFFLE] Target MACs:", macs)

#     hosts = resolve_hosts_from_macs(macs)
#     if not hosts:
#         print("[IP SHUFFLE] No hosts resolved from MACs")
#         return

#     ip_octets = build_ip_octets_for_hosts(hosts)

#     host_arg = ",".join(hosts)
#     ips_arg = ";".join(ip_octets)

#     # cmd = [
#     #     "sudo", "python3", IP_SHUFFLER_SCRIPT,
#     #     "--host", host_arg,
#     #     "--ips", ips_arg
#     # ]
#     cmd = [
#     "python3", IP_SHUFFLER_SCRIPT,
#     "--host", host_arg,
#     "--ips", ips_arg
#     ]

#     print("[IP SHUFFLE] Running:", " ".join(cmd))
#     result = subprocess.run(cmd, check=False)
#     print(f"[IP SHUFFLE] Exit code: {result.returncode}")


# def do_rrm(links, candidate_paths):
#     """
#     Runs one command like:
#       python3 rrm.py --specific_multiple --hosts "h1,h2" --opt "2"
#     """
#     print("[RRM] Avoid links:", links)

#     if not candidate_paths:
#         print("[RRM] No candidate safe path found")
#         return

#     print("[RRM] Candidate safe paths:")
#     for p in candidate_paths:
#         print(p)

#     chosen = candidate_paths[0]

#     host1 = chosen["host1"]
#     host2 = chosen["host2"]
#     opt = str(chosen["option_number"])

#     cmd = [
#         "python3", IP_SHUFFLER_SCRIPT,
#         "--host", host_arg,
#         "--ips", ips_arg
#     ]

#     print("[RRM] Running:", " ".join(cmd))
#     result = subprocess.run(cmd, check=False)
#     print(f"[RRM] Exit code: {result.returncode}")


# def dispatch_mitigation(final_obj, hoplist_csv):
#     decision = final_obj.get("final_decision", "do_nothing")
#     macs = final_obj.get("final_macs", [])
#     links = final_obj.get("final_links", [])

#     print("[DECISION]", decision)
#     print("MACs:", macs)
#     print("Links:", links)

#     candidate_paths = []
#     if decision in ["rrm", "both"]:
#         candidate_paths = path_selector(final_obj, hoplist_csv)

#     if decision == "do_nothing":
#         print("[ACTION] No action taken")

#     elif decision == "ip":
#         do_ip_shuffle(macs)

#     elif decision == "rrm":
#         do_rrm(links, candidate_paths)

#     elif decision == "both":
#         do_ip_shuffle(macs)
#         do_rrm(links, candidate_paths)

#     else:
#         print("[ERROR] Unknown decision:", decision)

In [ ]:
# from ip_shuffle_endpoint import ip_shuffle_endpoint
# from route_mutate_endpoint import route_shuffle_endpoint

# ip_shuffle_endpoint(
#     host=h1,
#     ips=1,
#     interval=15,
#     no_block_pid=True
# )

NameError: name 'h1' is not defined

In [ ]:
import random

from ip_shuffle_endpoint import ip_shuffle_endpoint
from route_mutate_endpoint import route_shuffle_endpoint
from collections import deque
import csv
import os
from mtd_utils import HostIPQueueManager, RouteHistoryManager, all_hosts
TOPOLOGY_FILE = "topology_s10.txt"

# import these from wherever you defined them
# from main_controller import ip_shuffle_endpoint, route_shuffle_endpoint


def parse_topology_hosts(topology_file):
    """
    Reads host entries like:
      h1, 10.0.0.1/24, 00:00:00:00:00:01

    Returns:
      mac_to_host = {
          "00:00:00:00:00:01": "h1",
          ...
      }
    """
    mac_to_host = {}

    with open(topology_file, "r") as f:
        for raw in f:
            line = raw.strip()

            if not line or line.startswith("#"):
                continue

            parts = [p.strip() for p in line.split(",")]

            if len(parts) == 3 and parts[0].startswith("h") and "/" in parts[1] and ":" in parts[2]:
                host = parts[0]
                mac = parts[2].upper()
                mac_to_host[mac] = host

    return mac_to_host


MAC_TO_HOST = parse_topology_hosts(TOPOLOGY_FILE)


def resolve_hosts_from_macs(macs):
    hosts = []

    for mac in macs:
        host = MAC_TO_HOST.get(mac.upper())
        if host:
            hosts.append(host)
        else:
            print(f"[WARN] MAC not found in topology: {mac}")

    return hosts


def build_ip_octets_for_hosts(hosts):
    """
    Example mapping:
      h1 -> 10
      h2 -> 20
      h3 -> 30
    """
    octets = []

    for h in hosts:
        try:
            num = int(h[1:])
            octet = str(num * 10)
            octets.append(octet)
        except Exception:
            print(f"[WARN] Could not derive octet for host {h}")
            octets.append(str(random.randint(10, 200)))

    return octets



def do_ip_shuffle(macs, ip_manager):
    print("[IP SHUFFLE] Target MACs:", macs)

    hosts = resolve_hosts_from_macs(macs)
    if not hosts:
        print("[IP SHUFFLE] No hosts resolved from MACs")
        return

    current_ips = ip_manager.get_current_ips()

    used_ips = set()
    for ip in current_ips.values():
        if ip is not None:
            used_ips.add(int(str(ip).split(".")[-1]))

    # free old IPs of selected hosts so they may change safely
    for h in hosts:
        old_ip = current_ips.get(h)
        if old_ip is not None:
            used_ips.discard(int(str(old_ip).split(".")[-1]))

    available_ips = [i for i in range(1, 256) if i not in used_ips]

    if len(available_ips) < len(hosts):
        print("[IP SHUFFLE] Not enough free IPs")
        return

    new_ips = random.sample(available_ips, len(hosts))
    shuffled_map = dict(zip(hosts, new_ips))

    host_arg = ",".join(hosts)
    ips_arg = ",".join(map(str, new_ips))

    print("[IP SHUFFLE] Calling endpoint:")
    print(f'  host="{host_arg}"')
    print(f'  ips="{ips_arg}"')

    ip_shuffle_endpoint(
        host=host_arg,
        ips=ips_arg,
        interval=15,
        no_block_pid=True
    )

    # push every host into deque every round
    for host in all_hosts:
        if host in shuffled_map:
            ip_manager.update_host_queue(host, shuffled_map[host])
        else:
            last_ip = ip_manager.current_ips.get(host)
            if last_ip is not None:
                ip_manager.update_host_queue(host, last_ip)

    ip_manager.save_to_csv()

    print("[IP SHUFFLE] Updated history:")
    print(ip_manager.get_all_host_ips())

    return shuffled_map #new

# def do_rrm(links, candidate_paths):
def do_rrm(links, candidate_paths, route_manager): # new 

    """
    Calls:
    route_shuffle_endpoint(
        specific_multiple=True,
        hosts="h1,h2;h1,h7",
        opt="3;4"
    )
    """
    print("[RRM] Avoid links:", links)

    if not candidate_paths:
        print("[RRM] No candidate safe path found")
        return

    print("[RRM] Candidate safe paths:")
    for p in candidate_paths:
        print(p)

    host_pairs = []
    opt_list = []

    for chosen in candidate_paths:
        host1 = chosen["host1"]
        host2 = chosen["host2"]
        opt = str(chosen["option_number"])

        route_manager.update_pair(host1, host2, opt) # new

        host_pairs.append(f"{host1},{host2}")
        opt_list.append(opt)

    hosts_arg = ";".join(host_pairs)
    opt_arg = ";".join(opt_list)

    print("[RRM] Calling endpoint:")
    print(f'  hosts="{hosts_arg}"')
    print(f'  opt="{opt_arg}"')

    route_shuffle_endpoint(
        specific_multiple=True,
        hosts=hosts_arg,
        opt=opt_arg
    )

    # new
    route_manager.save_to_csv()
    print("[RRM] Updated route history:")
    # new

    return candidate_paths # new

# def dispatch_mitigation(final_obj, hoplist_csv):
# def dispatch_mitigation(final_obj, hoplist_csv, ip_manager):
def dispatch_mitigation(final_obj, hoplist_csv, ip_manager, route_manager):
    decision = final_obj.get("final_decision", "do_nothing")
    macs = final_obj.get("final_macs", [])
    links = final_obj.get("final_links", [])

    print("[DECISION]", decision)
    print("MACs:", macs)
    print("Links:", links)

    candidate_paths = []
    ip_result = None
    route_result = None
    
    if decision in ["rrm", "both"]:
        candidate_paths = path_selector(final_obj, hoplist_csv)

    if decision == "do_nothing":
        print("[ACTION] No action taken")

    # elif decision == "ip":
    #     do_ip_shuffle(macs)
    elif decision == "ip":
        # do_ip_shuffle(macs, ip_manager)
        ip_result = do_ip_shuffle(macs, ip_manager) # new

    elif decision == "rrm":
        # do_rrm(links, candidate_paths)
        # do_rrm(links, candidate_paths , route_manager)  
        route_result = do_rrm(links, candidate_paths, route_manager) # new


    elif decision == "both":
        # do_ip_shuffle(macs)
        # do_ip_shuffle(macs, ip_manager)
        # do_rrm(links, candidate_paths)
        # do_rrm(links, candidate_paths , route_manager) # 
        ip_result = do_ip_shuffle(macs, ip_manager) # new
        route_result = do_rrm(links, candidate_paths, route_manager) # new

    else:
        print("[ERROR] Unknown decision:", decision)

    return ip_result, route_result

In [ ]:
from collections import deque
import csv
import os
import importlib
import mtd_utils
import pandas as pd
import json


from mtd_utils import HostIPQueueManager
# create ONCE
ip_manager = HostIPQueueManager()

route_manager = RouteHistoryManager(all_hosts, queue_size=10) # 
route_manager.load_from_csv() # new 

# initialize (only first time OR before loading)
for i in range(1, 41):
    ip_manager.set_host_ips(f"h{i}", [i])

# load previous history (VERY IMPORTANT)
ip_manager.load_from_csv()
print(ip_manager.get_current_ips())
print(ip_manager.get_all_host_ips())

final_obj = json.loads(final_out)
# dispatch_mitigation(final_obj, "hop_list.csv", ip_manager)
# dispatch_mitigation(final_obj, "hop_list.csv", ip_manager, route_manager) #
ip_result, route_result = dispatch_mitigation(final_obj, "hop_list.csv", ip_manager, route_manager) # new

decision_ts=last_ts_scene

HOST_CSV = "host_stats_onos.csv"
LINK_CSV = "link_stats_onos.csv"
EXAMPLES_JSONL = "examples.jsonl"

import time

def wait_for_next_two_ts(_prepare, csv_file, decision_ts):
    while True:
        df = _prepare(csv_file)
        ts = sorted(df["timestamp"].unique())
        future = [t for t in ts if t > decision_ts]

        if len(future) >= 2:
            return future[:2]

        time.sleep(2)



[LOAD] loading from ip_history.csv
{'h1': '127', 'h2': '82', 'h3': '3', 'h4': '4', 'h5': '5', 'h6': '6', 'h7': '7', 'h8': '8', 'h9': '9', 'h10': '10', 'h11': '11', 'h12': '12', 'h13': '13', 'h14': '14', 'h15': '15', 'h16': '16', 'h17': '17', 'h18': '18', 'h19': '19', 'h20': '20', 'h21': '21', 'h22': '22', 'h23': '23', 'h24': '24', 'h25': '25', 'h26': '26', 'h27': '27', 'h28': '28', 'h29': '29', 'h30': '30', 'h31': '31', 'h32': '32', 'h33': '33', 'h34': '34', 'h35': '35', 'h36': '36', 'h37': '37', 'h38': '38', 'h39': '39', 'h40': '40'}
{'h1': ['161', '161', '161', '161', '161', '161', '161', '161', '1', '127'], 'h2': ['90', '90', '90', '90', '90', '90', '90', '90', '2', '82'], 'h3': ['117', '117', '117', '117', '117', '117', '117', '117', '3', '3'], 'h4': ['8', '8', '8', '8', '8', '8', '8', '8', '4', '4'], 'h5': ['115', '115', '115', '115', '115', '115', '115', '115', '5', '5'], 'h6': ['159', '159', '159', '159', '159', '159', '159', '159', '6', '6'], 'h7': ['61', '61', '61', '61', '61'

In [ ]:
# final_obj = json.loads(final_out)
# # dispatch_mitigation(final_obj, "hop_list.csv", ip_manager)
# dispatch_mitigation(final_obj, "hop_list.csv", ip_manager, route_manager)


[DECISION] do_nothing
MACs: []
Links: []
[ACTION] No action taken


In [ ]:
# decision_ts=last_ts_scene

In [ ]:
# HOST_CSV = "host_stats_onos.csv"
# LINK_CSV = "link_stats_onos.csv"
# EXAMPLES_JSONL = "examples.jsonl"

# import time

# def wait_for_next_two_ts(_prepare, csv_file, decision_ts):
#     while True:
#         df = _prepare(csv_file)
#         ts = sorted(df["timestamp"].unique())
#         future = [t for t in ts if t > decision_ts]

#         if len(future) >= 2:
#             return future[:2]

#         time.sleep(2)



In [ ]:
#average last 5 instead of only last point!!
def validate_effect(scene, final_obj, decision_ts, host_csv=HOST_CSV, link_csv=LINK_CSV):
    def _prepare(csv_file):
        df = pd.read_csv(csv_file)
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
        return df.dropna(subset=["timestamp"]).copy()

    def mean2(vals):
        return float(sum(vals) / len(vals)) if vals else 0.0

    def weighted_mean(vals):
        if not vals:
            return 0.0
        weights = list(range(1, len(vals) + 1))   # older gets less, recent gets more
        wsum = sum(weights)
        return float(sum(w * x for w, x in zip(weights, vals)) / wsum)

    #replace below with different scoring for links # new
    # def selected_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean):
    #     red_pps = (before_pps - after_pps_mean) / max(before_pps, 1e-9)
    #     red_mbps = (before_mbps - after_mbps_mean) / max(before_mbps, 1e-9)
    #     return max(red_pps, red_mbps)

    # def other_candidate_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean):
    #     rise_pps = (after_pps_mean - before_pps) / max(before_pps, 1e-9)
    #     rise_mbps = (after_mbps_mean - before_mbps) / max(before_mbps, 1e-9)
    #     rise = max(rise_pps, rise_mbps)

    #     # improve => positive
    #     # rise up to 10% => tolerated
    #     # rise > 10% => negative proportional to excess
    #     if rise < 0:
    #         return abs(rise)
    #     elif rise <= 0.10:
    #         return 0.0
    #     else:
    #         return -(rise - 0.10)
    #replace above with different scoring for links # new

    def selected_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean, is_link=False):
        red_pps = (before_pps - after_pps_mean) / max(before_pps, 1e-9)
        red_mbps = (before_mbps - after_mbps_mean) / max(before_mbps, 1e-9)

        raw = max(red_pps, red_mbps)

        if is_link:
            # links usually improve less sharply, so soften expectation
            return raw * 1.35
        else:
            return raw
        
    def other_candidate_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean, is_link=False):
        rise_pps = (after_pps_mean - before_pps) / max(before_pps, 1e-9)
        rise_mbps = (after_mbps_mean - before_mbps) / max(before_mbps, 1e-9)
        rise = max(rise_pps, rise_mbps)

        if rise < 0:
            gain = abs(rise)
            return gain * (1.15 if is_link else 1.0)

        tolerance = 0.20 if is_link else 0.10
        if rise <= tolerance:
            return 0.0

        excess = rise - tolerance
        penalty = excess * (0.5 if is_link else 1.0)
        return -penalty
    # new above


    decision_ts = pd.to_datetime(decision_ts)
    # wait_for_next_two_ts(_prepare, host_csv, decision_ts)
    next_two_host = wait_for_next_k_ts(_prepare, host_csv, decision_ts,k=4)
    next_two_link = wait_for_next_k_ts(_prepare, link_csv, decision_ts,k=4)

    host_df = _prepare(host_csv)
    link_df = _prepare(link_csv)

    final_macs = [str(x).upper() for x in final_obj.get("final_macs", [])]
    final_links = [str(x) for x in final_obj.get("final_links", [])]

    candidate_host_stats = scene.get("flagged_host_stats", [])
    candidate_link_stats = scene.get("flagged_link_stats", [])

    host_candidate_map = {
        str(x.get("mac", "")).upper(): x
        for x in candidate_host_stats if x.get("mac")
    }
    link_candidate_map = {
        str(x.get("link_id", "")): x
        for x in candidate_link_stats if x.get("link_id")
    }

    candidate_macs = list(host_candidate_map.keys())
    candidate_links = list(link_candidate_map.keys())

    selected_macs_set = set(final_macs)
    selected_links_set = set(final_links)

    candidate_macs_set = set(candidate_macs)
    candidate_links_set = set(candidate_links)

    other_candidate_macs = sorted(candidate_macs_set - selected_macs_set)
    other_candidate_links = sorted(candidate_links_set - selected_links_set)

    selected_host_scores = []
    selected_link_scores = []
    other_candidate_host_scores = []
    other_candidate_link_scores = []

    host_trace_selected = []
    host_trace_other = []
    link_trace_selected = []
    link_trace_other = []

    # ---------------- Selected hosts ----------------
    for macu in final_macs:
        cand = host_candidate_map.get(macu)

        after_rows = host_df[
            (host_df["host_mac"].astype(str).str.upper() == macu) &
            (host_df["timestamp"].isin(next_two_host))
        ].sort_values("timestamp")

        if cand is None or after_rows.empty:
            continue

        before_pps = float(
            weighted_mean(cand["tx_pps_trend"]) +
            weighted_mean(cand["rx_pps_trend"])
        )
        before_mbps = float(
            weighted_mean(cand["tx_kbps_trend"]) +
            weighted_mean(cand["rx_kbps_trend"])
        ) / 1000.0

        after_pps_list = (
            after_rows["rx_pps"].astype(float).values +
            after_rows["tx_pps"].astype(float).values
        ).tolist()
        after_mbps_list = (
            after_rows["rx_mbps"].astype(float).values +
            after_rows["tx_mbps"].astype(float).values
        ).tolist()

        # after_pps_mean = mean2(after_pps_list)
        # after_mbps_mean = mean2(after_mbps_list)
        after_pps_mean = weighted_mean(after_pps_list) # new
        after_mbps_mean = weighted_mean(after_mbps_list) # new
        

        # score = selected_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean)
        score = selected_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean, is_link=False)
        selected_host_scores.append(score)

        host_trace_selected.append({
            "mac": macu,
            "before": {
                "pps": round(before_pps, 4),
                "mbps": round(before_mbps, 4)
            },
            "after_points": {
                "pps": [round(x, 4) for x in after_pps_list],
                "mbps": [round(x, 4) for x in after_mbps_list]
            },
            "score": round(score, 4)
        })

    # ---------------- Selected links ----------------
    for link_id in final_links:
        cand = link_candidate_map.get(link_id)

        after_rows = link_df[
            (link_df["link_id"].astype(str) == link_id) &
            (link_df["timestamp"].isin(next_two_link))
        ].sort_values("timestamp")

        if cand is None or after_rows.empty:
            continue

        before_pps = float(
            weighted_mean(cand["tx_pps_trend"]) +
            weighted_mean(cand["rx_pps_trend"])
        )
        before_mbps = float(
            weighted_mean(cand["tx_kbps_trend"]) +
            weighted_mean(cand["rx_kbps_trend"])
        ) / 1000.0

        after_pps_list = (
            after_rows["rx_pps"].astype(float).values +
            after_rows["tx_pps"].astype(float).values
        ).tolist()
        after_mbps_list = (
            after_rows["rx_mbps"].astype(float).values +
            after_rows["tx_mbps"].astype(float).values
        ).tolist()

        # after_pps_mean = mean2(after_pps_list)
        # after_mbps_mean = mean2(after_mbps_list)
        after_pps_mean = weighted_mean(after_pps_list) # new
        after_mbps_mean = weighted_mean(after_mbps_list) # new

        # score = selected_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean)
        score = selected_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean, is_link=True) # new
        selected_link_scores.append(score)

        link_trace_selected.append({
            "link_id": link_id,
            "before": {
                "pps": round(before_pps, 4),
                "mbps": round(before_mbps, 4)
            },
            "after_points": {
                "pps": [round(x, 4) for x in after_pps_list],
                "mbps": [round(x, 4) for x in after_mbps_list]
            },
            "score": round(score, 4)
        })

    # ---------------- Other candidate hosts ----------------
    for macu in other_candidate_macs:
        cand = host_candidate_map.get(macu)

        after_rows = host_df[
            (host_df["host_mac"].astype(str).str.upper() == macu) &
            (host_df["timestamp"].isin(next_two_host))
        ].sort_values("timestamp")

        if cand is None or after_rows.empty:
            continue

        before_pps = float(
            weighted_mean(cand["tx_pps_trend"]) +
            weighted_mean(cand["rx_pps_trend"])
        )
        before_mbps = float(
            weighted_mean(cand["tx_kbps_trend"]) +
            weighted_mean(cand["rx_kbps_trend"])
        ) / 1000.0

        after_pps_list = (
            after_rows["rx_pps"].astype(float).values +
            after_rows["tx_pps"].astype(float).values
        ).tolist()
        after_mbps_list = (
            after_rows["rx_mbps"].astype(float).values +
            after_rows["tx_mbps"].astype(float).values
        ).tolist()

        # after_pps_mean = mean2(after_pps_list)
        # after_mbps_mean = mean2(after_mbps_list)
        after_pps_mean = weighted_mean(after_pps_list)
        after_mbps_mean = weighted_mean(after_mbps_list)

        # score = other_candidate_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean)
        score = other_candidate_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean, is_link=False) # new
        other_candidate_host_scores.append(score)

        host_trace_other.append({
            "mac": macu,
            "before": {
                "pps": round(before_pps, 4),
                "mbps": round(before_mbps, 4)
            },
            "after_points": {
                "pps": [round(x, 4) for x in after_pps_list],
                "mbps": [round(x, 4) for x in after_mbps_list]
            },
            "score": round(score, 4)
        })

    # ---------------- Other candidate links ----------------
    for link_id in other_candidate_links:
        cand = link_candidate_map.get(link_id)

        after_rows = link_df[
            (link_df["link_id"].astype(str) == link_id) &
            (link_df["timestamp"].isin(next_two_link))
        ].sort_values("timestamp")

        if cand is None or after_rows.empty:
            continue

        before_pps = float(
            weighted_mean(cand["tx_pps_trend"]) +
            weighted_mean(cand["rx_pps_trend"])
        )
        before_mbps = float(
            weighted_mean(cand["tx_kbps_trend"]) +
            weighted_mean(cand["rx_kbps_trend"])
        ) / 1000.0

        after_pps_list = (
            after_rows["rx_pps"].astype(float).values +
            after_rows["tx_pps"].astype(float).values
        ).tolist()
        after_mbps_list = (
            after_rows["rx_mbps"].astype(float).values +
            after_rows["tx_mbps"].astype(float).values
        ).tolist()

        # after_pps_mean = mean2(after_pps_list)
        # after_mbps_mean = mean2(after_mbps_list)
        after_pps_mean = weighted_mean(after_pps_list)
        after_mbps_mean = weighted_mean(after_mbps_list)

        # score = other_candidate_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean)
        score = other_candidate_score_from_values(before_pps, before_mbps, after_pps_mean, after_mbps_mean, is_link=True) # new
        other_candidate_link_scores.append(score)

        link_trace_other.append({
            "link_id": link_id,
            "before": {
                "pps": round(before_pps, 4),
                "mbps": round(before_mbps, 4)
            },
            "after_points": {
                "pps": [round(x, 4) for x in after_pps_list],
                "mbps": [round(x, 4) for x in after_mbps_list]
            },
            "score": round(score, 4)
        })
    # commented for better scoring //
    # selected_scores = selected_host_scores + selected_link_scores
    # other_candidate_scores = other_candidate_host_scores + other_candidate_link_scores
    # all_scores = selected_scores + other_candidate_scores

    # selected_score = mean2(selected_scores)
    # other_candidate_score = mean2(other_candidate_scores)
    # final_score = mean2(all_scores)

    # if final_score >= 0.70:
    #     label = "good"
    # elif final_score >= 0.40:
    #     label = "neutral"
    # elif final_score == 0.0: # do_nothing case
    #     label = "neutral"
    # else:
    #     label = "bad"

    # if decision == "ip":
    #     result = "ip_shuffle improved the combined candidate situation" if label == "good" else "ip_shuffle gave limited or weak combined improvement"
    # elif decision == "rrm":
    #     result = "route_mutation improved the combined candidate situation" if label == "good" else "route_mutation gave limited or weak combined improvement"
    # elif decision == "both":
    #     result = "combined mitigation improved the candidate situation" if label == "good" else "combined mitigation gave limited or weak improvement"
    # else:
    #     result = "no mitigation applied"
    # commented above for better scoring //
    selected_scores = selected_host_scores + selected_link_scores
    other_candidate_scores = other_candidate_host_scores + other_candidate_link_scores

    selected_score = mean2(selected_scores)
    other_candidate_score = mean2(other_candidate_scores)

    decision = final_obj.get("final_decision", "")

    if decision == "do_nothing":
        final_score = 0.0
        label = "neutral"
        result = "no mitigation applied"

    else:
        # selected targets are the main signal
        # other candidates only provide a small adjustment
        final_score = selected_score + 0.25 * other_candidate_score

        if final_score >= 0.70:
            label = "good"
        elif final_score >= 0.20:
            label = "neutral"
        else:
            label = "bad"
    
    if decision == "ip":
        result = "ip_shuffle improved the selected targets" if label == "good" else "ip_shuffle gave limited or weak improvement"
    elif decision == "rrm":
        result = "route_mutation improved the selected targets" if label == "good" else "route_mutation gave limited or weak improvement"
    elif decision == "both":
        result = "combined mitigation improved the selected targets" if label == "good" else "combined mitigation gave limited or weak improvement"
    else:
        result = "unknown decision"

    # old code cont.
    return {
        "label": label,
        "result": result,
        "score": round(final_score, 4),
        "selected_score": round(selected_score, 4),
        "other_candidate_score": round(other_candidate_score, 4),
        "validation_trace": {
            "selected_hosts": host_trace_selected,
            "selected_links": link_trace_selected,
            "other_candidate_hosts": host_trace_other,
            "other_candidate_links": link_trace_other
        }
    }

#average last 5 instead of only last point!!



# new compatible to validate_effect
def make_example(scene, final_obj, validation_out):
    return {
        "candidates": {
            "host_stats": scene.get("flagged_host_stats", []),
            "link_stats": scene.get("flagged_link_stats", [])
        },
        "decision": {
            "final_decision": final_obj.get("final_decision"),
            "final_macs": final_obj.get("final_macs", []),
            "final_links": final_obj.get("final_links", [])
        },
        "validation_trace": validation_out.get("validation_trace", {}),
        "outcome": {
            "result": validation_out["result"],
            "label": validation_out["label"],
            "score": validation_out["score"],
            "selected_score": validation_out["selected_score"],
            "other_candidate_score": validation_out["other_candidate_score"]
        }
    }

# new compatible to validate_effect

In [37]:
def save_example(example, filename=EXAMPLES_JSONL):
    with open(filename, "a", encoding="utf-8") as f:
        f.write(json.dumps(example) + "\n")

In [26]:
scenex

{'host_report': {'decision': 'ip_shuffle',
  'macs_to_shuffle': ['00:00:00:00:00:1B', '00:00:00:00:00:25'],
  'confidence': 0.9,
  'observation': [{'mac': '00:00:00:00:00:1B', 'reason': 'tx spike'},
   {'mac': '00:00:00:00:00:25', 'reason': 'tx spike'}]},
 'link_report': {'decision': 'reroute',
  'links_to_avoid': ['of:000b:3 -> of:000d:2', 'of:000c:3 -> of:000d:3'],
  'confidence': 0.85,
  'observation': [{'link_id': 'of:000b:3 -> of:000d:2',
    'reason': 'sustained high tx_pps and rx_kbps spikes'},
   {'link_id': 'of:000c:3 -> of:000d:3',
    'reason': 'continuous high tx_pps and rx_kbps over multiple points'}]},
 'flagged_host_stats': [{'mac': '00:00:00:00:00:1B',
   'tx_pps_trend': [3.79, 1.67, 1.89, 1.85, 1.85, 1.49, 0.6, 0.7, 0.6, 0.7],
   'rx_pps_trend': [2.1, 0.15, 0.15, 0.15, 0.15, 0.1, 0.0, 0.0, 0.0, 0.0],
   'tx_kbps_trend': [3.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0],
   'rx_kbps_trend': [2.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]},
  {'mac': '00:00:00:00:00:

In [68]:
    # # new judge feature for later 
    # def make_example(scene, final_obj, validation_out, judge_out):
    #     return {
    #         "scene_at_decision": {
    #             "flagged_host_stats": scene.get("flagged_host_stats", []),
    #             "flagged_link_stats": scene.get("flagged_link_stats", [])
    #         },
    #         "decision": final_obj.get("final_decision"),
    #         "result": validation_out["result"],
    #         "label": validation_out["label"],
    #         "llm_judge": {
    #             "observation": judge_out.get("observation", ""),
    #             "score": float(judge_out.get("score", 0.0)),
    #             "llm_label": judge_out.get("label", "neutral")
    #         }
    #     }

    # import json

    # def build_judge_prompt(scene, final_obj, validation_out):
    #     payload = {
    #         "scene_at_decision": {
    #             "flagged_host_stats": scene.get("flagged_host_stats", []),
    #             "flagged_link_stats": scene.get("flagged_link_stats", [])
    #         },
    #         "decision": final_obj.get("final_decision"),
    #         "validation_result": validation_out.get("result", ""),
    #         "validation_label": validation_out.get("label", "")
    #     }

    #     prompt = f"""
    # You are a post-decision network mitigation judge.

    # A mitigation decision has already been made and executed.
    # Your task is to review the case and provide:
    # 1. a short observation,
    # 2. a score from 0.0 to 1.0,
    # 3. an llm label: good, bad, or neutral.

    # Judge using:
    # - the suspicious host and link scene at decision time,
    # - the decision that was taken,
    # - the validation result after execution.

    # Scoring:
    # - 0.00 to 0.30 = bad
    # - 0.31 to 0.69 = neutral
    # - 0.70 to 1.00 = good

    # Rules:
    # - Be strict and evidence-based.
    # - Do not invent facts.
    # - Use only the provided input.
    # - Keep the observation short and factual.
    # - Return JSON only.
    # - Return exactly these keys:
    # observation, score, label

    # Input:
    # {json.dumps(payload, indent=2)}

    # Return exactly:
    # {{
    # "observation": "short factual explanation",
    # "score": 0.0,
    # "label": "good"
    # }}
    # """
    #     return prompt.strip()

    # import json

    # def get_llm_judge(scene, final_obj, validation_out):
    #     prompt = build_judge_prompt(scene, final_obj, validation_out)
    #     raw = call_cloud_llm_judge(prompt, "kimi-k2-thinking:cloud")

    #     if isinstance(raw, dict):
    #         judge_out = raw
    #     else:
    #         try:
    #             judge_out = json.loads(raw)
    #         except Exception:
    #             judge_out = {
    #                 "observation": "judge output could not be parsed",
    #                 "score": 0.0,
    #                 "label": "neutral"
    #             }

    #     score = float(judge_out.get("score", 0.0))
    #     score = max(0.0, min(1.0, score))

    #     label = str(judge_out.get("label", "neutral")).strip().lower()
    #     if label not in {"good", "bad", "neutral"}:
    #         label = "neutral"

    #     return {
    #         "observation": str(judge_out.get("observation", "")).strip(),
    #         "score": score,
    #         "label": label
    #     }

    # '''    "llm_judge": {
    #     "observation": "ip_shuffle was appropriate because the post-decision validation supports reduced issue severity.",
    #     "score": 0.91,
    #     "llm_label": "good"
    # }'''
    # # new judge feature for later 

## validation

In [69]:
# # new judge feature for later 
# judge_out = get_llm_judge(scene, final_obj, validation_out)
# example = make_example(scene, final_obj, validation_out, judge_out)
# # new judge feature for later 

In [27]:
decision_ts=last_ts_scene
# validation_out = validate_effect(final_obj, decision_ts)
validation_out = validate_effect(scenex,final_obj, decision_ts)
example = make_example(scenex, final_obj, validation_out)
save_example(example)
print(example)

KeyboardInterrupt: 

In [ ]:
# # after validation
# update_trace(host_trace, flagged_hosts, selected_hosts, outcome_label)
# update_trace(link_trace, flagged_links, selected_links, outcome_label)

# # save
# save_trace_csv("host_trace.csv", host_trace)
# save_trace_csv("link_trace.csv", link_trace)

In [71]:
decision_ts

Timestamp('2026-04-04 00:39:53.701348')

In [72]:
# 21.21

In [ ]:
import json

def fetch_best_good_and_worst_bad(example_jsonl_path):
    rows = []

    with open(example_jsonl_path, "r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                obj["_line_no"] = line_no
                rows.append(obj)
            except json.JSONDecodeError:
                continue

    good_rows = [r for r in rows if r.get("outcome", {}).get("label") == "good"]
    bad_rows  = [r for r in rows if r.get("outcome", {}).get("label") == "bad"]

    best_good = max(good_rows, key=lambda x: x.get("outcome", {}).get("score", float("-inf")), default=None)
    worst_bad = min(bad_rows, key=lambda x: x.get("outcome", {}).get("score", float("inf")), default=None)

    def build_output(obj):
        if obj is None:
            return None

        host_candidates = {h["mac"]: h for h in obj.get("candidates", {}).get("host_stats", []) if "mac" in h}
        link_candidates = {l["link_id"]: l for l in obj.get("candidates", {}).get("link_stats", []) if "link_id" in l}

        vt = obj.get("validation_trace", {})

        selected_hosts = []
        for h in vt.get("selected_hosts", []):
            mac = h.get("mac")
            base = host_candidates.get(mac, {})
            selected_hosts.append({
                "mac": mac,
                "tx_pps_trend": base.get("tx_pps_trend", []),
                "rx_pps_trend": base.get("rx_pps_trend", []),
                "tx_kbps_trend": base.get("tx_kbps_trend", []),
                "rx_kbps_trend": base.get("rx_kbps_trend", []),
                "after_pps": h.get("after_points", {}).get("pps", []),
                "after_mbps": h.get("after_points", {}).get("mbps", [])
            })

        selected_links = []
        for l in vt.get("selected_links", []):
            link_id = l.get("link_id")
            base = link_candidates.get(link_id, {})
            selected_links.append({
                "link_id": link_id,
                "tx_pps_trend": base.get("tx_pps_trend", []),
                "rx_pps_trend": base.get("rx_pps_trend", []),
                "tx_kbps_trend": base.get("tx_kbps_trend", []),
                "rx_kbps_trend": base.get("rx_kbps_trend", []),
                "after_pps": l.get("after_points", {}).get("pps", []),
                "after_mbps": l.get("after_points", {}).get("mbps", [])
            })

        other_hosts = []
        for h in vt.get("other_candidate_hosts", []):
            mac = h.get("mac")
            base = host_candidates.get(mac, {})
            other_hosts.append({
                "mac": mac,
                "tx_pps_trend": base.get("tx_pps_trend", []),
                "rx_pps_trend": base.get("rx_pps_trend", []),
                "tx_kbps_trend": base.get("tx_kbps_trend", []),
                "rx_kbps_trend": base.get("rx_kbps_trend", []),
                "after_pps": h.get("after_points", {}).get("pps", []),
                "after_mbps": h.get("after_points", {}).get("mbps", [])
            })

        other_links = []
        for l in vt.get("other_candidate_links", []):
            link_id = l.get("link_id")
            base = link_candidates.get(link_id, {})
            other_links.append({
                "link_id": link_id,
                "tx_pps_trend": base.get("tx_pps_trend", []),
                "rx_pps_trend": base.get("rx_pps_trend", []),
                "tx_kbps_trend": base.get("tx_kbps_trend", []),
                "rx_kbps_trend": base.get("rx_kbps_trend", []),
                "after_pps": l.get("after_points", {}).get("pps", []),
                "after_mbps": l.get("after_points", {}).get("mbps", [])
            })

        return {
            "line_no": obj["_line_no"],
            "final_decision": obj.get("decision", {}).get("final_decision"),
            "selected_candidates": {
                "hosts": selected_hosts,
                "links": selected_links
            },
            "other_candidates": {
                "hosts": other_hosts,
                "links": other_links
            }
        }

    return {
        "good_example": build_output(best_good),
        "bad_example": build_output(worst_bad)
    }
result = fetch_best_good_and_worst_bad("examples.jsonl")
print(json.dumps(result, separators=(",", ":")))

{
  "good_example": {
    "line_no": 6,
    "final_decision": "both",
    "selected_candidates": {
      "hosts": [
        {
          "mac": "00:00:00:00:00:0B",
          "tx_pps_trend": [
            0.69,
            0.59,
            0.6,
            0.7,
            0.6,
            0.64,
            0.75,
            37739.8,
            27336.96,
            20661.26
          ],
          "rx_pps_trend": [
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.49,
            0.7,
            0.4
          ],
          "tx_kbps_trend": [
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            1.0,
            449847.0,
            325840.0,
            246271.0
          ],
          "rx_kbps_trend": [
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
            0.0,
      

In [5]:
# import json
# import time

# # -----------------------------
# # Load valid rows from examples
# # -----------------------------
# rows = []
# with open("examples.jsonl", "r", encoding="utf-8") as f:
#     for line in f:
#         line = line.strip()
#         if not line:
#             continue
#         try:
#             rows.append(json.loads(line))
#         except Exception:
#             continue

# # -----------------------------
# # Recent window
# # -----------------------------
# WINDOW = 10
# recent = rows[-WINDOW:]

# # -----------------------------
# # Compact fields only
# # -----------------------------
# compact = []
# for r in recent:
#     compact.append({
#         "candidates": r.get("candidates", {}),
#         "decision": r.get("decision", {}),
#         "validation_trace": r.get("validation_trace", {}),
#         "outcome": r.get("outcome", {})
#     })

# prompt = f"""
# You are an SDN incident report writer.

# You are given recent mitigation records.
# Write them as short factual case reports, similar to a crime/incident report.

# Your job:
# For each record, describe:
# 1. What was observed
#    - which hosts or links showed abnormal or stressed behavior
# 2. What was attempted (ONLY if meaningful)
#    - mention the action only if it impacted the situation
# 3. What happened after
#    - whether selected targets improved
#    - whether selected targets stayed problematic
#    - whether results were mixed
# 4. What the final outcome was
#    - good, neutral, or bad
# 5. What short lesson this case suggests

# Interpretation rules:
# - score > 0 means improved
# - score <= 0 means still problematic
# - selected_* are mitigated targets
# - other_candidate_* are flagged but not selected
# - a record can have improved targets but still a bad overall outcome
# - if both positive and non-positive scores appear, say "mixed results"
# - ignore mtd_action completely

# FOCUS RULES:
# - Do NOT mention do_nothing unless it is important to explain the outcome
# - Prioritize describing:
#   - what was abnormal
#   - what changed (or didn’t)
#   - whether the action actually helped
# - Highlight when:
#   - a path or link change did NOT improve the situation
#   - some targets improved but overall outcome remained bad
# - Keep emphasis on effectiveness, not on listing actions

# WRITING STYLE:
# - Short, factual, incident-style sentences
# - No speculation beyond the data
# - No overgeneralization
# - Use concrete wording like:
#   - "two links were rerouted but only one improved"
#   - "host traffic dropped but link congestion remained"
#   - "no intervention occurred and some candidates improved naturally"

# Return ONLY valid JSON.

# Output schema:
# {{
#   "case_reports": [
#     {{
#       "case_id": 1,
#       "incident_observed": "what looked abnormal",
#       "what_happened": "what was done and what changed",
#       "outcome": "good | neutral | bad",
#       "lesson": "what this case teaches"
#     }}
#   ],
#   "overall_pattern": "short factual pattern across cases",
#   "key_findings": [
#     "short factual finding",
#     "short factual finding",
#     "short factual finding"
#   ]
# }}

# Data:
# {json.dumps(compact, ensure_ascii=False)}

# Return JSON only.
# """.strip()

# out, latency = call_cloud_llm(prompt)

# print(out)
# print("Latency:", latency)

In [ ]:
# def build_llm_history_summary(examples_jsonl=EXAMPLES_JSONL, n=5):
#     if not os.path.exists(examples_jsonl):
#         return ""

#     rows = []
#     try:
#         with open(examples_jsonl, "r", encoding="utf-8") as f:
#             for line in f:
#                 line = line.strip()
#                 if not line:
#                     continue
#                 try:
#                     rows.append(json.loads(line))
#                 except Exception:
#                     continue
#     except Exception as e:
#         print("[WARN] could not read examples:", e)
#         return ""

#     if not rows:
#         return ""

#     recent = rows[-n:]

#     compact = []
#     for r in recent:
#         compact.append({
#             "decision": r.get("decision", {}),
#             "mtd_action": r.get("mtd_action", {}),
#             "outcome": r.get("outcome", {})
#         })

#     prompt = f"""
# You are analyzing the recent history of an SDN mitigation system.

# You are given the last {n} decision records. Each record contains:
# - decision: what was decided (final_decision, final_macs, final_links)
# - mtd_action: what action was actually executed (ip_changes, route_changes)
# - outcome: what happened after (label, score, result)

# Your job:
# - Summarize what patterns you see across these decisions.
# - Note which MACs and links keep appearing.
# - Note whether actions are working (good), neutral, or failing (bad).
# - Clearly state which specific decision and action gave a good outcome if any.
# - Note if the same action is being repeated without improvement.
# - Keep your summary concise and factual, in MAC and link ID language only.
# - Do not use host names like h2 or h39.

# Return ONLY valid JSON in this exact schema:
# {{
#   "pattern": "short description of overall pattern",
#   "repeated_macs": ["MAC_ADDRESS"],
#   "repeated_links": ["LINK_ID"],
#   "working": "what decision/action gave good outcome and why",
#   "not_working": "what is not working if anything",
#   "suggestion": "what the next decision should consider"
# }}

# Data:
# {json.dumps(compact, ensure_ascii=False)}

# Return JSON only.
# """.strip()

#     try:
#         out, latency = call_cloud_llm(prompt)
#         print(f"[HISTORY SUMMARY] latency={latency:.2f}s")
#         parsed = json.loads(out)
#         return json.dumps(parsed, ensure_ascii=False)
#     except Exception as e:
#         print("[WARN] history summary failed:", e)
#         return ""

In [4]:
import json
import time

# -----------------------------
# Load valid rows from examples
# -----------------------------
rows = []
with open("examples.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        try:
            rows.append(json.loads(line))
        except Exception:
            continue

# -----------------------------
# Recent window
# -----------------------------
WINDOW = 5
recent = rows[-WINDOW:]

# -----------------------------
# Compact fields only
# -----------------------------
compact = []
for r in recent:
    compact.append({
        "decision": r.get("decision", {}),
        "mtd_action": r.get("mtd_action", {}),
        "outcome": r.get("outcome", {})
    })

prompt = f"""
You are analyzing the recent history of an SDN mitigation system.

You are given the last {WINDOW} decision records. Each record contains:
- decision: what was decided (final_decision, final_macs, final_links)
- mtd_action: what action was actually executed (ip_changes, route_changes)
- outcome: what happened after (label, score, result)

Your job:
- Summarize what patterns you see across these decisions.
- Note which MACs and links keep appearing.
- Note whether actions are working (good), neutral, or failing (bad).
- Note if the same action is being repeated without improvement.
- Keep your summary concise and factual, in MAC and link ID language only.
- Do not use host names like h2 or h39.

Return ONLY valid JSON in this exact schema:
{{
  "pattern": "short description of overall pattern",
  "repeated_macs": ["MAC_ADDRESS"],
  "repeated_links": ["LINK_ID"],
  "working": "what is working if anything",
  "not_working": "what is not working if anything",
  "suggestion": "what the next decision should consider"
}}

Data:
{json.dumps(compact, ensure_ascii=False)}

Return JSON only.
""".strip()

out, latency = call_cloud_llm(prompt)
print(out)
print("Latency:", latency)

{"pattern":"Repeated MACs 00:00:00:00:00:02 and 00:00:00:00:00:26 appear in all decisions with the same core links; route changes on h2-h38 give limited or neutral improvement.","repeated_macs":["00:00:00:00:00:02","00:00:00:00:00:26"],"repeated_links":["of:0000000000000002:1 -> of:000000000000000b:4","of:0000000000000007:1 -> of:000000000000000d:5","of:000000000000000b:3 -> of:000000000000000d:2","of:0000000000000007:2 -> of:000000000000000a:3"],"working":"None","not_working":"Route changes targeting h2-h38 with these MACs and links repeatedly fail to improve scores","suggestion":"Try route changes on other links such as of:000000000000000a:1 -> of:000000000000000d:6 or of:000000000000000b:2 -> of:000000000000000c:2, and consider adding different MACs to diversify the set."}
Latency: 16.365670680999756
